<div style="background-color: #f8f9fa; padding: 25px; border-radius: 10px; border-left: 6px solid #1a365d; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">
    <h1 style="color: #1a365d; margin-bottom: 5px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">Data Science Assignment 5</h1>
    <h3 style="color: #4a5568; margin-top: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">University of Tehran - Spring 2026</h3>
    <hr style="border: 1px solid #e2e8f0;">
    <p style="font-size: 16px; color: #2d3748;">
        <b>Course:</b> Data Science <br><br>
        <b>🧑‍💻 Implementer:</b> Mohammad Reza Pirhadi <br>
        <b>🆔 Student ID:</b> 810102423 <br><br>
        <h5 style="color: #1a365d;"> TASK 1 Implementation</h5>
    </p>
</div>

<div class="alert alert-info" style="border-left: 4px solid #3182ce; background-color: #ebf8ff; color: #2b6cb0;">
    <strong>💡 Project Overview:</strong> 
        This notebook contains the programmatic implementation for Assignment 5 - Task 1. It focuses on predicting Formula 1 pit stop strategies using historical race datasets. The implementation covers a complete machine learning pipeline, including feature engineering, class imbalance handling, training classification baselines, and implementing Semi-Supervised Pseudo-Labeling and Active Learning Uncertainty Sampling loops to iteratively leverage unlabeled data.
</div>

## Table of Contents
1. [Exploratory Data Analysis (EDA) & Feature Engineering](#eda-preprocessing)
2. [Baseline Supervised Models & Class Imbalance Handling](#baselines)
3. [Semi-Supervised Learning (Pseudo-Labeling Loop)](#ssl-pseudo-labeling)
4. [Active Learning & Uncertainty Query Strategies](#active-learning)
5. [Evaluation, Strategy Comparison & Trade-off Analysis](#evaluation)

<a id="eda-preprocessing"></a>
<h2 style="color: #5490d9; border-bottom: 2px solid #cbd5e0; padding-bottom: 5px; margin-top: 40px;">1. Exploratory Data Analysis (EDA) & Feature Engineering</h2>

<div class="alert alert-warning" style="background-color: #fffaf0; border-left: 4px solid #dd2a20; color: #c02121;">
    <strong>🚗 Methodology:</strong>
    This phase involves parsing historical F1 CSV files, performing statistical checks on race features, and engineering status-driven attributes (such as tire degradation cycles and relative gaps). To prevent data leakage, <code>safety_car_likely</code> and <code>stop_duration_ms</code> are dropped. Numerical variables are scaled using <code>StandardScaler</code>, categorical dimensions are encoded, and the data is stratified into an <strong>80/10/10</strong> train/validation/test split.
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp, entropy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from itertools import cycle

sns.set_theme(style="whitegrid", palette="muted")
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def load_and_engineer_features(races_path, pit_stops_path, lap_times_path):
    print("Loading and Engineering Features...")
    # بارگذاری داده‌ها (در محیط واقعی مسیر فایل‌های خود را قرار دهید)
    races = pd.read_csv(races_path)
    pit_stops = pd.read_csv(pit_stops_path)
    lap_times = pd.read_csv(lap_times_path)
    
    # فیلتر کردن فصل‌های 2014 تا 2024
    races = races[(races['year'] >= 2014) & (races['year'] <= 2024)]
    
    # ادغام جداول
    df = pd.merge(pit_stops, races[['raceId', 'year', 'name']], on='raceId', how='inner')
    df = pd.merge(df, lap_times, on=['raceId', 'driverId', 'lap'], how='inner')
    
    # --- ساخت ویژگی‌های پایه و محاسباتی ---
    # مرتب‌سازی برای محاسبه دقیق شیفت‌ها
    df = df.sort_values(by=['raceId', 'driverId', 'lap'])
    
    df['lap_time_delta'] = df.groupby(['raceId', 'driverId'])['milliseconds'].diff().fillna(0)
    df['laps_since_last_pit'] = df.groupby(['raceId', 'driverId']).cumcount() + 1
    df['laps_remaining'] = 70 - df['lap'] # فرض میانگین 70 دور برای هر مسابقه
    
    # شبیه‌سازی ویژگی‌هایی که نیاز به دیتای تله‌متری سنگین دارند (جهت اجرای پروژه)
    np.random.seed(42)
    df['approx_gap_ahead'] = np.random.uniform(0.0, 15.0, len(df))
    df['approx_gap_behind'] = np.random.uniform(0.0, 15.0, len(df))
    df['safety_car_likely'] = np.random.choice([0, 1], p=[0.9, 0.1], size=len(df))
    
    # --- ساخت ۲ ویژگی ابتکاری (Heuristics) ---
    # 1. شاخص تخریب لاستیک (ترکیب مسافت طی شده با افت زمان دور)
    df['tire_degradation_index'] = df['laps_since_last_pit'] * df['lap_time_delta'].apply(lambda x: max(0, x))
    
    # 2. فشار رقابتی (هرچه فاصله با ماشین جلویی و عقبی کمتر باشد، فشار بیشتر است)
    df['competitive_pressure'] = 1.0 / (df['approx_gap_ahead'] + df['approx_gap_behind'] + 0.1)
    
    print(f"Dataset shape after feature engineering: {df.shape}")
    return df

# df_raw = load_and_engineer_features('races.csv', 'pit_stops.csv', 'lap_times.csv')

In [ ]:
def apply_heuristics(row):
    """تخصیص استراتژی‌ها بر اساس منطق مسابقات F1"""
    if row['safety_car_likely'] == 1:
        return 'Emergency'
    elif row['approx_gap_ahead'] < 2.0 and row['lap_time_delta'] < 0:
        return 'Undercut'
    elif row['approx_gap_behind'] < 2.0 and row['lap_time_delta'] > 0:
        return 'Overcut'
    else:
        return 'Standard'

def label_data(df):
    print("Applying Rule-based Labeling...")
    df['strategy'] = df.apply(apply_heuristics, axis=1)
    print("Class Distribution:")
    print(df['strategy'].value_counts(normalize=True) * 100)
    return df

# df_labeled = label_data(df_raw)

In [ ]:
def split_datasets(df, target_col='strategy'):
    print("Splitting Data into Labeled (10%) and Unlabeled (90%)...")
    
    # حذف ویژگی‌هایی که باعث تقلب مدل (Leakage) می‌شوند
    columns_to_drop = ['safety_car_likely', 'duration', 'stop_duration_ms', 'time', 'time_x', 'time_y']
    df_clean = df.drop(columns=[c for c in columns_to_drop if c in df.columns], errors='ignore')
    
    X = df_clean.drop(columns=[target_col])
    
    # برای مدل‌سازی فقط از ستون‌های عددی استفاده می‌کنیم
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    X = X[numeric_cols]
    
    y = df_clean[target_col]
    
    # تقسیم 10 درصد لیبل‌دار و 90 درصد بدون لیبل
    X_labeled, X_unlabeled, y_labeled, y_unlabeled_true = train_test_split(
        X, y, test_size=0.90, stratify=y, random_state=42
    )
    
    # تقسیم 80-20 روی داده‌های لیبل‌دار برای آموزش مدل پایه
    X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
        X_labeled, y_labeled, test_size=0.20, stratify=y_labeled, random_state=42
    )
    
    return X_train_base, X_test_base, y_train_base, y_test_base, X_unlabeled, y_unlabeled_true, df_clean

# X_train, X_test, y_train, y_test, X_unlabeled, y_unlabeled_true, df_clean = split_datasets(df_labeled)

In [ ]:
def perform_eda(df_clean, X_train, X_unlabeled, y_train):
    print("--- Exploratory Data Analysis (EDA) ---")
    
    # 1. توزیع کلاس‌ها در داده‌های آموزش
    plt.figure(figsize=(8, 4))
    sns.countplot(x=y_train, order=['Standard', 'Undercut', 'Emergency', 'Overcut'])
    plt.title("Class Distribution in Labeled Data (10%)")
    plt.show()

    # 2. بررسی کیفیت تقسیم با آزمون KS برای ویژگی‌های کلیدی
    key_features = ['lap_time_delta', 'laps_since_last_pit', 'approx_gap_ahead', 'tire_degradation_index']
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    print("Kolmogorov-Smirnov Test Results (p-value > 0.05 is good):")
    for i, feature in enumerate(key_features):
        if feature in X_train.columns:
            ax = axes[i//2, i%2]
            sns.kdeplot(X_train[feature].dropna(), label='Labeled', ax=ax, fill=True, alpha=0.4)
            sns.kdeplot(X_unlabeled[feature].dropna(), label='Unlabeled', ax=ax, fill=True, alpha=0.4)
            ax.set_title(f'Distribution of {feature}')
            ax.legend()
            
            stat, p_val = ks_2samp(X_train[feature].dropna(), X_unlabeled[feature].dropna())
            print(f"{feature:25} | KS Stat: {stat:.4f} | P-value: {p_val:.4f}")
    plt.tight_layout()
    plt.show()

    # 3. هیت‌مپ همبستگی
    plt.figure(figsize=(10, 8))
    sns.heatmap(df_clean.select_dtypes(include=[np.number]).corr(), 
                annot=False, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Feature Correlation Heatmap')
    plt.show()

# perform_eda(df_clean, X_train, X_unlabeled, y_train)

<a id="baselines"></a>
<h2 style="color: #5490d9; border-bottom: 2px solid #cbd5e0; padding-bottom: 5px; margin-top: 40px;">2. Baseline Supervised Models & Class Imbalance Handling</h2>

<div class="alert alert-warning" style="background-color: #fffaf0; border-left: 4px solid #dd2a20; color: #c02121;">
    <strong>🚗 Methodology:</strong>
    We train and validate three diverse estimators: <code>Logistic Regression</code>, <code>Random Forest</code>, and <code>Gradient Boosting</code>. Due to severe class imbalance (where emergency pit stops are rare compared to standard configurations), cost-sensitive learning is integrated using the <code>class_weight='balanced'</code> parameter to penalize minority-class errors heavily during optimization.
</div>


In [ ]:
def train_supervised_baseline(X_train, y_train, X_test, y_test):
    print("Training Supervised Baselines...")
    
    models = {
        'Logistic Regression': LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(class_weight='balanced', max_depth=6, ccp_alpha=0.01, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
    }
    
    best_f1 = 0
    best_model_name = ""
    best_pipeline = None
    
    for name, model in models.items():
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', model)
        ])
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)
        f1 = f1_score(y_test, preds, average='macro')
        print(f"{name} - Macro F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            best_model_name = name
            best_pipeline = pipeline
            
    print(f"\nBest Model: {best_model_name}")
    
    # گزارش و ماتریس درهم‌ریختگی بهترین مدل
    best_preds = best_pipeline.predict(X_test)
    print("\nClassification Report:\n", classification_report(y_test, best_preds))
    
    cm = confusion_matrix(y_test, best_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
    plt.title(f"Confusion Matrix - {best_model_name}")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # رسم Feature Importance (برای مدل‌های درختی)
    if hasattr(best_pipeline.named_steps['classifier'], 'feature_importances_'):
        importances = best_pipeline.named_steps['classifier'].feature_importances_
        indices = np.argsort(importances)[::-1][:10] # 10 ویژگی برتر
        plt.figure(figsize=(8, 4))
        plt.bar(range(10), importances[indices], align="center")
        plt.xticks(range(10), X_train.columns[indices], rotation=45, ha='right')
        plt.title("Top 10 Feature Importances")
        plt.tight_layout()
        plt.show()
        
    return best_pipeline

# best_base_model = train_supervised_baseline(X_train, y_train, X_test, y_test)

<a id="ssl-pseudo-labeling"></a>
<h2 style="color: #5490d9; border-bottom: 2px solid #cbd5e0; padding-bottom: 5px; margin-top: 40px;">3. Semi-Supervised Learning (Pseudo-Labeling Loop)</h2>

<div class="alert alert-warning" style="background-color: #fffaf0; border-left: 4px solid #dd2a20; color: #c02121;">
    <strong>🚗 Methodology:</strong>
    We implement a self-training loop over five iterations. The base model generates predictions on the unlabeled pool using <code>predict_proba()</code>. Samples exceeding confidence thresholds of <code>0.60</code>, <code>0.75</code>, and <code>0.85</code> are assigned pseudo-labels and merged back into the active training set, measuring the impact of noise injection vs. data expansion.
</div>


In [ ]:
def run_pseudo_labeling(base_pipeline, X_train, y_train, X_unlabeled, y_unlabeled_true, X_test, y_test):
    print("Running Pseudo Labeling...")
    thresholds = [0.60, 0.75, 0.85]
    rounds = 5
    results = {}
    
    for threshold in thresholds:
        print(f"\n--- Threshold: {threshold} ---")
        X_train_curr, y_train_curr = X_train.copy(), y_train.copy()
        X_pool = X_unlabeled.copy()
        
        # مدل را Clone می‌کنیم تا از ابتدا آموزش ببیند
        from sklearn.base import clone
        model = clone(base_pipeline)
        
        f1_scores = []
        
        for r in range(rounds):
            if len(X_pool) == 0: break
                
            model.fit(X_train_curr, y_train_curr)
            probs = model.predict_proba(X_pool)
            max_probs = np.max(probs, axis=1)
            preds = model.predict(X_pool)
            
            confident_idx = np.where(max_probs >= threshold)[0]
            
            if len(confident_idx) < 50:
                print(f"Round {r+1}: Only {len(confident_idx)} samples. Stopping early.")
                break
                
            # اضافه کردن داده‌های جدید
            pseudo_X = X_pool.iloc[confident_idx]
            pseudo_y = pd.Series(preds[confident_idx], index=pseudo_X.index)
            
            X_train_curr = pd.concat([X_train_curr, pseudo_X])
            y_train_curr = pd.concat([y_train_curr, pseudo_y])
            X_pool = X_pool.drop(pseudo_X.index)
            
            # ارزیابی روی Test Set ثابت
            curr_f1 = f1_score(y_test, model.predict(X_test), average='macro')
            f1_scores.append(curr_f1)
            print(f"Round {r+1} | Added: {len(confident_idx)} | Macro F1: {curr_f1:.4f}")
            
        results[threshold] = f1_scores
        
    plt.figure(figsize=(8, 5))
    for t, scores in results.items():
        plt.plot(range(1, len(scores)+1), scores, marker='o', label=f'Threshold {t}')
    plt.title('Pseudo Labeling F1-Score Progression')
    plt.xlabel('Round')
    plt.ylabel('Macro F1 Score (on Test Set)')
    plt.legend()
    plt.grid(True)
    plt.show()

# run_pseudo_labeling(best_base_model, X_train, y_train, X_unlabeled, y_unlabeled_true, X_test, y_test)

<a id="active-learning"></a>
<h2 style="color: #5490d9; border-bottom: 2px solid #cbd5e0; padding-bottom: 5px; margin-top: 40px;">4. Active Learning & Uncertainty Query Strategies</h2>

<div class="alert alert-warning" style="background-color: #fffaf0; border-left: 4px solid #dd2a20; color: #c02121;">
    <strong>🚗 Methodology:</strong>
    We construct an active learning loop executing 8 query cycles. In each step, the model queries 20 samples from the unlabeled pool based on three distinct uncertainty metrics: <code>Least Confidence</code>, <code>Margin</code>, and <code>Entropy</code>. The selected samples receive simulated ground-truth labels and are added to the training set to optimize parameter updates.
</div>


In [ ]:
def run_active_learning(base_pipeline, X_train, y_train, X_unlabeled, y_unlabeled_true, X_test, y_test):
    print("Running Active Learning...")
    strategies = ['Least Confidence', 'Margin', 'Entropy']
    rounds = 8
    query_size = 20
    al_results = {}
    
    for strategy in strategies:
        print(f"\n--- Strategy: {strategy} ---")
        X_train_curr, y_train_curr = X_train.copy(), y_train.copy()
        X_pool, y_pool_true = X_unlabeled.copy(), y_unlabeled_true.copy()
        
        from sklearn.base import clone
        model = clone(base_pipeline)
        f1_scores, labeled_sizes = [], []
        
        for r in range(rounds):
            model.fit(X_train_curr, y_train_curr)
            probs = model.predict_proba(X_pool)
            
            if strategy == 'Least Confidence':
                uncertainty = 1.0 - np.max(probs, axis=1)
            elif strategy == 'Margin':
                sorted_probs = np.sort(probs, axis=1)
                uncertainty = 1.0 - (sorted_probs[:, -1] - sorted_probs[:, -2])
            elif strategy == 'Entropy':
                uncertainty = entropy(probs, axis=1)
                
            # انتخاب سمپل‌های نامطمئن
            query_indices = np.argsort(uncertainty)[-query_size:]
            queried_X = X_pool.iloc[query_indices]
            queried_y = y_pool_true.iloc[query_indices] # شبیه‌سازی اوراکل
            
            X_train_curr = pd.concat([X_train_curr, queried_X])
            y_train_curr = pd.concat([y_train_curr, queried_y])
            X_pool = X_pool.drop(queried_X.index)
            y_pool_true = y_pool_true.drop(queried_y.index)
            
            curr_f1 = f1_score(y_test, model.predict(X_test), average='macro')
            f1_scores.append(curr_f1)
            labeled_sizes.append(len(X_train_curr))
            
            print(f"Round {r+1} | Total Labeled: {len(X_train_curr)} | Macro F1: {curr_f1:.4f}")
            
        al_results[strategy] = {'sizes': labeled_sizes, 'scores': f1_scores}
        
    plt.figure(figsize=(8, 5))
    for s, data in al_results.items():
        plt.plot(data['sizes'], data['scores'], marker='s', label=s)
    plt.title('Active Learning - Learning Curves')
    plt.xlabel('Number of Labeled Samples')
    plt.ylabel('Macro F1 Score (on Test Set)')
    plt.legend()
    plt.grid(True)
    plt.show()

# run_active_learning(best_base_model, X_train, y_train, X_unlabeled, y_unlabeled_true, X_test, y_test)

<a id="evaluation"></a>
<h2 style="color: #5490d9; border-bottom: 2px solid #cbd5e0; padding-bottom: 5px; margin-top: 40px;">5. Evaluation, Strategy Comparison & Trade-off Analysis</h2>

<div class="alert alert-warning" style="background-color: #fffaf0; border-left: 4px solid #dd2a20; color: #c02121;">
    <strong>🚗 Methodology:</strong>
    A comprehensive comparative analysis is conducted using macro-averaged F1-scores, precision-recall curves, and confusion matrices. We plot performance trajectory curves across iterations to analyze the efficiency trade-offs between semi-supervised self-labeling (which is cost-free but noisy) and active learning (which is high-accuracy but resource-intensive).
</div>


In [ ]:
def final_evaluation_and_comparison(best_base_model, best_pseudo_model, best_al_model, 
                                    X_test, y_test, 
                                    base_f1, pseudo_history, al_history):
    """
    best_base_model: بهترین مدل آموزش دیده روی 10% داده‌ها
    best_pseudo_model: مدل نهایی خروجی گرفته شده از بهترین آستانه Pseudo Labeling
    best_al_model: مدل نهایی خروجی گرفته شده از بهترین استراتژی Active Learning
    base_f1: مقدار F1-Score مدل پایه
    pseudo_history: لیست تاریخچه F1 بهترین آستانه در Pseudo Labeling (مثلا آستانه 0.75)
    al_history: دیکشنری نتایج Active Learning شامل sizes و scores برای بهترین استراتژی (مثلا Entropy)
    """
    
    print("--- 1. Summary Comparison Table ---")
    # ساخت جدول خلاصه
    best_pseudo_f1 = max(pseudo_history)
    best_al_f1 = max(al_history['scores'])
    
    summary_data = {
        'Method': ['Supervised Baseline (10%)', 'Pseudo Labeling (Best)', 'Active Learning (Best)'],
        'Macro F1-Score': [base_f1, best_pseudo_f1, best_al_f1],
        'Data Requirement': ['Low (Only initial 10%)', 'High Unlabeled Pool', 'Low Labeled + Human Oracle'],
        'Labeling Cost': ['None', 'None (Self-labeled)', 'High (Requires human intervention)']
    }
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df) # اگر در جوپیتر هستید جدول زیبا نمایش داده می‌شود
    print("\n")

    print("--- 2. ROC Curves & AUC Comparison ---")
    # آماده‌سازی لیبل‌ها برای حالت Multi-class
    classes = ['Standard', 'Undercut', 'Emergency', 'Overcut']
    y_test_bin = label_binarize(y_test, classes=classes)
    n_classes = y_test_bin.shape[1]
    
    models = {
        'Baseline': best_base_model,
        'Pseudo Labeling': best_pseudo_model,
        'Active Learning': best_al_model
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = cycle(['blue', 'red', 'green', 'orange'])
    
    for ax, (model_name, model) in zip(axes, models.items()):
        if model is None:
            ax.set_title(f"{model_name} (Model not provided)")
            continue
            
        # پیش‌بینی احتمالات
        y_score = model.predict_proba(X_test)
        
        # محاسبه ROC برای هر کلاس
        for i, color in zip(range(n_classes), colors):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=color, lw=2, 
                    label=f'{classes[i]} (AUC = {roc_auc:.2f})')
            
        ax.plot([0, 1], [0, 1], 'k--', lw=2)
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC - {model_name}')
        ax.legend(loc="lower right", fontsize='small')
        
    plt.tight_layout()
    plt.show()

    print("--- 3. Comprehensive Learning Curves (Trade-off Analysis) ---")
    # فرض: اندازه اولیه داده‌های لیبل دار
    initial_labeled_size = al_history['sizes'][0] - 20 # چون در راند اول ۲۰ تا اضافه شده
    
    plt.figure(figsize=(10, 6))
    
    # خط ثابت مدل پایه
    plt.axhline(y=base_f1, color='red', linestyle='--', label=f'Baseline F1 ({base_f1:.4f})')
    
    # منحنی Pseudo Labeling (محور X بر اساس راند است، آن را به سایز تقریبی داده تبدیل میکنیم یا راندها را مقایسه میکنیم)
    # برای سادگی مقایسه، از تعداد نمونه‌های اضافه شده (X-axis) استفاده میکنیم
    # چون در AL سمپل‌های مشخص (۲۰ تا) اضافه کردیم، محور X را روی سایز لیبل دارها تنظیم میکنیم
    
    plt.plot(al_history['sizes'], al_history['scores'], marker='s', color='blue', linewidth=2, label='Active Learning (Entropy)')
    
    # برای Pseudo Labeling، فرض میکنیم در هر راند تعداد مشخصی اضافه شده (مثلا 50 تا) 
    # در محیط واقعی باید سایز دقیق را از خروجی تابع pseudo labeling پاس بدهید.
    pseudo_sizes = [initial_labeled_size + (i * 50) for i in range(len(pseudo_history))]
    plt.plot(pseudo_sizes, pseudo_history, marker='o', color='green', linewidth=2, label='Pseudo Labeling (Thresh=0.75)')
    
    plt.title('Performance vs. Number of Labeled Samples (Trade-off)')
    plt.xlabel('Number of Labeled Samples Used in Training')
    plt.ylabel('Macro F1-Score on Test Set')
    plt.legend()
    plt.grid(True)
    plt.show()

# نحوه اجرای این تابع:
# شما باید مدل‌های نهایی خروجی گرفته شده از مراحل قبل را به همراه لیست امتیازاتشان به این تابع پاس بدهید.
# مثال:
# final_evaluation_and_comparison(
#     best_base_model=best_base_model, 
#     best_pseudo_model=best_model_from_pseudo, # مدلی که از بهترین حلقه پسودو استخراج شده
#     best_al_model=best_model_from_al,         # مدلی که از بهترین حلقه اکتیو لرنینگ استخراج شده
#     X_test=X_test, 
#     y_test=y_test, 
#     base_f1=0.65,                               # مقدار واقعی F1 پایه
#     pseudo_history=[0.66, 0.68, 0.67, 0.67, 0.68], # خروجی دیکشنری pseudo (مثلا برای threshold 0.75)
#     al_history={'sizes': [100, 120, 140, 160, 180], 'scores': [0.65, 0.69, 0.73, 0.75, 0.76]} # خروجی اکتیو لرنینگ
# )